# V2 训练数据：候选设计与原图检索

当前先逐算子 review；编辑原图来源、循环编排仍待讨论。已知循环续跑会额外重算部分阶段，尚未定稿。线上 RAG 后续单独设计。

固定文章／视觉发布 → 统一准备构题材料 → 知识／参考／目标共同设计 →（仅编辑：搜原图、核验、定稿）→ 校验与审核 → 校验已选训练输入 → 目标检索／审核 → 导出。

**输入不再包含手写plan、intent、选材ID；候选目标由流程自动检索。**全局seed、预算、题型、模型及声明的原图检索源属于运行配置。任务由知识、条件与视觉推理支撑；简单直接的知识应用也可成立，明确执行要求只作辅助，以新题实际检查信号、题面和原判据的对应，不要求training-free提升。未发布知识不会进入模型；配图、编辑原图、监督目标分别管理。已有知识本身仍有质量限制，因此新产物称开发候选。

从下面第一个代码cell开始；默认只读实际保存结果。所有展示使用Markdown、表格和原生图片，不使用HTML。逐步输入／输出与prompt见 [training/stepbystep.ipynb](stepbystep.ipynb)。

训练目标候选数通过 target_candidates_per_task 配置；同一冻结题目可逐张核验多张独立目标，只有通过目标审核的记录才能导出。引用ID缺口保留审计，不提前阻止寻找目标。


V2：知识／参考＋隔离的候选目标共同构题。training_materials 统一读取可选知识、视觉材料和候选目标，列出每个目标可用的参考编号，design_candidates 可见候选目标但不将其作为知识证据；任务冻结后逐张目标审核。目标不进入训练输入，只有唯一目标项 loss=True。尚无 V2 真实训练或模型质量结论。

**当前按知识应用与候选目标共同构题思路迭代。** 新题用于检查材料、任务、公开展示条件与原题判据的对应；失败可用于研究训练，但不等于正确监督已备齐。训练输入与题目共同选定并绑定；评测仍可独立使用公开题面检索。旧案例仍按冻结版本解释；新版执行使用新的NEW_RUN。

当前主要prompt：[候选设计](prompts/design_candidates.md)、[原图审核（本地与外搜共用）](prompts/select_edit_source.md)、[编辑定稿](prompts/construct.md)、[任务审核](prompts/review_task.md)、[训练目标审核](prompts/review_target.md)。逐步版每个模型算子前展示当前prompt全文；旧案例的实际输入以其冻结请求为准。

In [ ]:
from curation.preparation.records import rows
from pathlib import Path
import sys
_candidates = [Path.cwd(), *Path.cwd().parents, Path("/yzp/zhaozy/yangzepeng/0905/demiwtg")]
PROJECT = next((p for p in _candidates if (p / "curation/training/authoring.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("找不到包含curation的项目根目录")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
try:
    from curation.preparation.inspection import show_records, show_summary, show_cases, snapshot_ref
    from curation.training.runtime import config as build_config
    from curation.preparation.records import run_records, run_state, rows
except ModuleNotFoundError as error:
    raise RuntimeError("请选择demiwtg内核：/yzp/zhaozy/yangzepeng/0905/env/bin/python") from error

MODE = "view_saved"  # execute运行；view_saved只读已有checkpoint。
BASE = PROJECT / "curation/training/runs"
SAVED_RUN = BASE / "pipeline_v2_training_review"
NEW_RUN = PROJECT / "curation/training/runs/pipeline_v2_training_target_loop_review_20260921"  # 输入／代码／prompt变化须新run。
knowledge_runs = []  # 可空：知识文本不是强制依据。
visual_runs = []  # 独立视觉发布 ID 或固定 DatasetRef；两类输入不能同时为空。
# 知识输入为最终交付；编辑原图检索的全库清单及外部源在下面config中公开配置。
# V2 训练构题可看独立候选目标；目标不进入知识证据或作答输入。
MODEL_BACKEND = "offline"  # local用本地Qwen；offline为相同prompt／context生成绑定请求。
# concepts=None使用全部发布概念；可传名称列表做可复现小批。
config = build_config(MODEL_BACKEND, concepts=None, seed=0, max_units=2, tasks_per_unit=1, training_sample_goal=2,
    reference_batch_size=4, max_target_cycles=2, max_training_attempts=100, max_context_chars=100000,
    scene_search={"image_ref": None,
                  "external_providers": ["commons"]},
    author_model="gpt-6-astra" if MODEL_BACKEND == "offline" else None,
    author_effort="high" if MODEL_BACKEND == "offline" else None)
CASE_ID = None  # 默认展示本阶段第一条实际参与处理的记录；也可填unit_id或task_id。
SHOW_AUDIT = False  # 完整来源、字段、实际模型请求；图片按角色原样展示。
THROUGH = "export"
if MODE not in {"view_saved", "execute"}:
    raise ValueError("MODE必须是view_saved或execute")
run = SAVED_RUN if MODE == "view_saved" else NEW_RUN
if MODE == "view_saved":
    saved = run_records(run).get("manifest")
    if saved is not None:
        config = saved["config"]
    else:
        print("请指定已有 V2 Lance run；后续查看单元需要已完成的阶段。")
print("项目：", PROJECT, "\n内核：", sys.executable, "\n模式：", MODE, "\n运行：", run)


print("本次运行配置：", config)

print("当前入口只读取 V2 Lance 结果。历史查看册保存在归档与 reviews 中。")


## 完整算子链

| 顺序 | 算子 | 输入 → 输出 |
|---|---|---|
| 1. 准备构题材料 | `map(PrepareTrainingMaterials)` | 固定文章／视觉发布 → 可选依据、候选目标、逐目标可用参考编号；不预分用途 |
| 2. 一次设计候选任务 | `filter（公开seed预算）→ map_prompt_async("design_candidates")` | 知识／参考＋候选目标＋全局预算 → 候选考点与任务设计 |
| 3. 展开候选与绑定证据 | `flat_map(ExpandCandidates)` | 模型候选列表 → 每题一行 |
| 4. 编辑：按初始场景搜本地图 | `map_async(SearchLocalScenes)` | 编辑原图需求及查询词＋全库图片清单 → 候选原图 |
| 5. 仅编辑：核验原图 | `map_prompt_async("select_edit_source")` | 知识考点、最终知识、候选原图像素与来源 → 合格原图或needs_edit_source |
| 6. 编辑：必要时外部补搜 | `map_async(SearchExternalScenes)` | 本地无候选或像素审核拒绝 → 外部候选或可追溯缺口 |
| 7. 编辑：核验外部原图 | `map_prompt_async("select_edit_source_external")` | 外部候选像素及来源 → 可用原图或缺口 |
| 8. 编辑：据真实原图定稿 | `map_prompt_async("construct")` | 候选设计＋选定原图像素 → 具体题面和判据 |
| 9. 校验任务契约 | `map(ValidateTask)` | 草稿和材料编号 → 绑定稳定知识ID的判据及冻结任务哈希 |
| 10. 审核任务质量 | `map_prompt_async("review_task")` | 草稿、全部构题证据、编辑原图 → 五项语义审核 |
| 11. 校验并组装已选训练输入 | `map(BindTrainingInputs)` | 题目绑定的参考／文本 → 一致性、隔离校验及实际输入；不重新选材 |
| 12. 寻找监督目标 | `flat_map(SelectTargetCandidate.expand)` | 冻结题面与声明的素材来源 → 查找与已绑定参考兼容的目标、待验目标或needs_target |
| 13. 核验训练目标 | `map_prompt_async("review_target")` | 冻结任务、依据、原图、候选目标及作答材料 → 目标审核 |
| 14. 导出并保留缺口 | `map(export_record) → filter` | 任务及各阶段审核 → ready／incomplete |

下面就是CLI实际执行的函数；map负责业务，map_prompt_async负责模型调用，checkpoint保存各边界。offline按同一请求生成待响应状态；不暗中继承本次聊天。

In [ ]:
from functools import partial
from demiflow.standalone import local_data
from curation.preparation.records import run_lock
from curation.training.runtime import graph_version
from curation.training.authoring import (AuthoringRunFiles, input_records, split_guard,
    knowledge_items, SelectTargetCandidate)
from curation.training.candidates import ExpandCandidates
from curation.training.scene_search import SceneSearch, SearchLocalScenes, SearchExternalScenes
from curation.training.scene_assets import ExistingImagePool
from curation.training.materials import PrepareTrainingMaterials, design_concepts
from curation.training.operators import ValidateTask, BindTrainingInputs, export_record
from curation.training.prompting import (prompt_config, prompt_responses,
    prepare_design, apply_design, prepare_external_edit_source, apply_external_edit_source,
    prepare_edit_source, apply_edit_source,
    prepare_construct, apply_construct, prepare_task_review, apply_task_review,
    prepare_target_review, apply_target_review)


from curation.training.loop import (training_targets, training_attempt,
    loop_progress, advance_progress, loop_stop, attempt_waiting)


In [ ]:
def run_training_attempt(scope, knowledge, files, data, run, config, guard, pack, target_pool,
                         through="export", prefix="", task_prefix=None):
    """One visible Dataset chain, shared by the loop and step-by-step review."""
    designed = (scope
        .map(partial(prepare_design, run=run, pack=pack))
        .map_prompt_async("design_candidates", config="tasks.yaml",
            inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
            output="design_candidates_result", call_output="design_candidates_call", error_output="design_candidates_error",
            when=lambda r: r["status"] == "knowledge_available", concurrency=1, queue_depth=1)
        .map(partial(apply_design, run=run))
        )
    designed = files.lance_checkpoint(designed, prefix + "design", extra=prompt_responses(run, "design_candidates", task_prefix))
    if through == "design":
        return None
    candidates = (designed.flat_map(ExpandCandidates(guard, config))
        )
    candidates = files.lance_checkpoint(candidates, prefix + "candidates")
    if through == "candidates":
        return None
    scene_search = SceneSearch(run, knowledge.flat_map(knowledge_items).take_all(), guard, config)
    edit_candidates = (candidates.map_async(SearchLocalScenes(scene_search), concurrency=1)
        )
    edit_candidates = files.lance_checkpoint(edit_candidates, prefix + "edit_search")
    if through == "edit_search":
        return None
    source_ready = (edit_candidates
        .map(partial(prepare_edit_source, run=run, pack=pack))
        .map_prompt_async("select_edit_source", config="tasks.yaml",
            inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
            output="select_edit_source_result", call_output="select_edit_source_call", error_output="select_edit_source_error",
            when=lambda r: r["status"] == "edit_candidates_ready", concurrency=1, queue_depth=1)
        .map(partial(apply_edit_source, run=run))
        )
    source_ready = files.lance_checkpoint(source_ready, prefix + "edit_source", extra=prompt_responses(run, "select_edit_source", task_prefix))
    if through == "edit_source":
        return None
    external_candidates = (source_ready.map_async(SearchExternalScenes(scene_search), concurrency=1)
        )
    external_candidates = files.lance_checkpoint(external_candidates, prefix + "edit_external_search")
    if through == "edit_external_search":
        return None
    all_sources = (external_candidates
        .map(partial(prepare_external_edit_source, run=run, pack=pack))
        .map_prompt_async("select_edit_source_external", config="tasks.yaml",
            inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
            output="select_edit_source_external_result", call_output="select_edit_source_external_call", error_output="select_edit_source_external_error",
            when=lambda r: r["status"] == "external_edit_candidates_ready", concurrency=1, queue_depth=1)
        .map(partial(apply_external_edit_source, run=run))
        )
    all_sources = files.lance_checkpoint(all_sources, prefix + "edit_external_source", extra=prompt_responses(run, "select_edit_source_external", task_prefix))
    if through == "edit_external_source":
        return None
    constructed = (all_sources
        .map(partial(prepare_construct, run=run, pack=pack))
        .map_prompt_async("construct", config="tasks.yaml",
            inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
            output="construct_result", call_output="construct_call", error_output="construct_error",
            when=lambda r: r["status"] == "selected", concurrency=1, queue_depth=1)
        .map(partial(apply_construct, run=run))
        )
    constructed = files.lance_checkpoint(constructed, prefix + "construct", extra=prompt_responses(run, "construct", task_prefix))
    if through == "construct":
        return None
    validated = (constructed.map(ValidateTask(guard))
        )
    validated = files.lance_checkpoint(validated, prefix + "validate")
    if through == "validate":
        return None
    reviewed = (validated
        .map(partial(prepare_task_review, run=run, pack=pack))
        .map_prompt_async("review_task", config="tasks.yaml",
            inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
            output="review_task_result", call_output="review_task_call", error_output="review_task_error",
            when=lambda r: r["status"] == "valid_task", concurrency=1, queue_depth=1)
        .map(partial(apply_task_review, run=run))
        )
    reviewed = files.lance_checkpoint(reviewed, prefix + "review", extra=prompt_responses(run, "review_task", task_prefix))
    if through == "review":
        return None
    bound_inputs = (reviewed.map(BindTrainingInputs(guard))
        )
    bound_inputs = files.lance_checkpoint(bound_inputs, prefix + "bind_inputs")
    if through == "bind_inputs":
        return None
    targets = (bound_inputs.flat_map(SelectTargetCandidate(guard, target_pool, limit=1 if task_prefix else config.get("target_candidates_per_task", 1)).expand)
        )
    targets = files.lance_checkpoint(targets, prefix + "targets")
    if through == "targets":
        return None
    target_reviewed = (targets
        .map(partial(prepare_target_review, run=run, pack=pack))
        .map_prompt_async("review_target", config="tasks.yaml",
            inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
            output="review_target_result", call_output="review_target_call", error_output="review_target_error",
            when=lambda r: r["status"] == "target_attached", concurrency=1, queue_depth=1)
        .map(partial(apply_target_review, run=run))
        )
    target_reviewed = files.lance_checkpoint(target_reviewed, prefix + "review_targets", extra=prompt_responses(run, "review_target", task_prefix))
    if through == "review_targets":
        return None
    exported = (target_reviewed.map(export_record)
        )
    exported = files.lance_checkpoint(exported, prefix + "export")
    return exported


In [ ]:
def run_pipeline(run, knowledge_runs, config, through="export", visual_runs=None):
    """Task-local material selection; the notebook is the CLI graph source."""
    with run_lock(run):
        if through in {"knowledge", "retrieve"}:
            raise ValueError("Training stages use training_materials and bind_inputs; use a new run")
        files = AuthoringRunFiles(run, knowledge_runs, "training", config, graph_version("training"), visual_runs=visual_runs)
        guard = split_guard(files)
        pack, options = prompt_config(run, config)
        data = local_data(prompt_packs={"tasks.yaml": pack}, prompt_options=options,
                          max_prompt_requests=config["model"]["max_calls"])
        # 急切校验发布冲突（同概念多篇不同文章＝显式错误，不被执行器包装）
        input_rows = list(input_records(files))
        selected_concepts = design_concepts(input_rows, config)
        target_pool = ExistingImagePool(run, guard, config)
        knowledge = (data.from_iter(lambda: iter(input_rows))
            .map(PrepareTrainingMaterials(files.knowledge_version, target_pool, config, guard, selected_concepts))
            )
        knowledge = files.lance_checkpoint(knowledge, "training_materials")
        if through == "training_materials":
            return files.finish()
        history = []
        progress = loop_progress()
        if config['training_design'] == 'knowledge_first':
            # Explicit research control; the target-first loop below is the default.
            scope = knowledge.filter(lambda r: r["authoring_selected"] and r["status"] != "needs_materials")
            exported = run_training_attempt(scope, knowledge, files, data, run, config, guard, pack, target_pool, through)
            if exported is None:
                return files.finish()
            history = exported.take_all()
            progress['accepted_samples'] = sum(bool(r.get('export_ready')) for r in history)
            stop_reason = 'knowledge_first_control_complete'
        else:
            target_visits = training_targets(knowledge.take_all(), config)
            while not (stop_reason := loop_stop(progress, len(target_visits), config)):
                attempt = training_attempt(target_visits, progress, config)
                prefix = '' if progress['attempts'] == 0 else f"attempt_{progress['attempts'] + 1:06d}_"
                scope = data.from_iter(lambda: iter([attempt]))
                scope = files.lance_checkpoint(scope, prefix + 'attempt_materials')
                if through == 'attempt_materials':
                    return files.finish()
                exported = run_training_attempt(scope, knowledge, files, data, run, config, guard, pack, target_pool,
                                                through, prefix, attempt['unit_id'])
                if exported is None:
                    return files.finish()
                results = exported.take_all()
                if len(results) != 1:
                    raise ValueError('A target visit must produce exactly one attempt result')
                history.extend(results)
                if attempt_waiting(results):
                    stop_reason = 'waiting_for_response_or_call_recovery'
                    break
                progress = advance_progress(progress, results[0], len(target_visits))
            # Only aggregate after actual attempts; this also commits valid empty output.
            if len(history) != 1:
                exported = data.from_iter(lambda: iter(history))
                exported = files.lance_checkpoint(exported, "export")
        ready = (exported.filter(lambda r: r["export_ready"])
            )
        ready = files.lance_checkpoint(ready, "ready")
        incomplete = (exported.filter(lambda r: not r["export_ready"])
            )
        incomplete = files.lance_checkpoint(incomplete, "incomplete")
        state = files.finish()
        state['training_progress'] = {**progress, 'stop_reason': stop_reason,
                                      'sample_goal': config['training_sample_goal']}
        files.records.put('latest', state, immutable=False)
        return state


## 执行与运行概览

In [ ]:
if MODE == "execute":
    import asyncio
    state = await asyncio.to_thread(run_pipeline, run, knowledge_runs, config, through=THROUGH, visual_runs=visual_runs)
show_summary(run)


## 实际候选与保留的失败

候选设计、题目、搜索记录、图片角色和审核逐条呈现；未入选概念可回knowledge查看，不混成坏题。

In [ ]:
show_cases(run)


## 完整请求与原始记录

In [ ]:
if SHOW_AUDIT:
    state = run_state(run)
    last = "export" if "export" in state["stages"] else next(reversed(state["stages"]))
    show_records(local_data().from_iter(lambda: rows(snapshot_ref(run, last))), last, CASE_ID, audit=True)
